In [9]:
import requests
import json
import zipfile
import io
from tqdm import tqdm
import pandas as pd
from pathlib import Path

In [2]:
with open("./data/survey_dataset_urls.json", "r", encoding="utf-8") as f:
    survey_dataset_urls = json.load(f)

In [18]:
distributions = []
idx_val = 0
for idx, url in tqdm(survey_dataset_urls.items()):
    suffix_ds = url.split("/")[-1].lower()
    try:
        resp = requests.get(
            f"https://data.europa.eu/api/hub/search/datasets/{suffix_ds}"
        )
        distributions.extend(resp.json()["result"]["distributions"])
    except Exception as e:
        print(f"Error scraping {idx}: {e}")

 40%|███▉      | 395/994 [00:54<01:19,  7.51it/s]

Error scraping 2019: Expecting value: line 1 column 1 (char 0)


 42%|████▏     | 414/994 [00:57<01:11,  8.13it/s]

Error scraping 2034: Expecting value: line 1 column 1 (char 0)


100%|██████████| 994/994 [02:25<00:00,  6.85it/s]


In [ ]:
with open("./data/distributions_metadata.json", "w", encoding="utf-8") as f:
    json.dump(distributions, f, indent=2)

## download the excel files

In [3]:
with open("./data/distributions_metadata.json", "r", encoding="utf-8") as f:
    distributions = json.load(f)

In [4]:
len(distributions)

6191

In [16]:
distributions_df = pd.json_normalize(distributions, sep="_")

distributions_df["access_url"] = distributions_df["access_url"].apply(lambda x: x[0])

no_nandistributions_df = distributions_df.dropna(subset=["title_fr"])
volume_b_docs_df = no_nandistributions_df[
    (
        (no_nandistributions_df["title_fr"].str.contains("volume_B."))
        | (no_nandistributions_df["title_fr"].str.contains("volume B."))
    )
    & ~(no_nandistributions_df["title_fr"].str.contains("volume_BP"))
]

In [18]:
# Create folder for downloads
downloads_folder = Path("./data/volume_b_docs")
downloads_folder.mkdir(exist_ok=True)

# Download each file (Excel or zip)
for _, row in tqdm(
    volume_b_docs_df.iterrows(), total=len(volume_b_docs_df), desc="Downloading"
):
    url = row["access_url"]

    # Get extension from title_fr (e.g., "volume_B.xls" -> "xls", "volume_B.zip" -> "zip")
    extension = row["title_fr"].split(".")[-1].lower()
    if extension not in ("xls", "xlsx", "zip"):
        extension = "xlsx"  # Default fallback

    # Remove extension before sanitizing, then add it back
    title_without_ext = row["title_fr"].rsplit(".", 1)[0]
    sanitized_title = (
        title_without_ext.replace(" ", "_")
        .replace("/", "_")
        .replace(":", "_")
        .replace(".", "_")
    )

    filename = f"{sanitized_title}.{extension}"
    filepath = downloads_folder / filename

    # Skip if already downloaded (for non-zip) or if extracted files exist
    if filepath.exists():
        continue

    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()

        if extension == "zip":
            # Unzip and extract contents to the downloads folder
            with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
                for member in zf.namelist():
                    # Skip directories and hidden files
                    if member.endswith("/") or member.startswith("__"):
                        continue
                    # Extract file to downloads folder with flat structure
                    member_filename = Path(member).name
                    extract_path = downloads_folder / member_filename
                    if not extract_path.exists():
                        with zf.open(member) as src, open(extract_path, "wb") as dst:
                            dst.write(src.read())
        else:
            with open(filepath, "wb") as f:
                f.write(resp.content)
    except Exception as e:
        print(f"Error downloading {url}: {e}")


Downloading: 100%|██████████| 131/131 [04:44<00:00,  2.17s/it]
